In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/antorio/FlashVSR_plus
%cd FlashVSR_plus

In [ ]:
# install torch dari index pytorch
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

# install sisanya dari pypi
!pip install -r requirements.txt

In [ ]:
!rm -rf ./models/FlashVSR-v1.1
!git lfs clone https://huggingface.co/JunhaoZhuang/FlashVSR-v1.1 ./models/FlashVSR-v1.1

In [ ]:
BASE_DIR = "/content/drive/MyDrive/c"
INPUT = "SR08_2.mp4" # @param {type:"string"}
MODE = "tiny" # @param ["full", "tiny", "tiny-long"]
SCALE = 2
OUTPUT_DIR = BASE_DIR + "/enh_output"

In [ ]:
import os
import subprocess

CHUNK_DIR = "/content/chunks"
CHUNK_SECONDS = 20

os.makedirs(CHUNK_DIR, exist_ok=True)

subprocess.run([
    "ffmpeg",
    "-i", f"{BASE_DIR}/{INPUT}",
    "-c", "copy",
    "-map", "0",
    "-segment_time", str(CHUNK_SECONDS),
    "-f", "segment",
    "-reset_timestamps", "1",
    f"{CHUNK_DIR}/part_%04d.mp4"
], check=True)

print("Video split selesai")

In [ ]:
import glob
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
chunks = sorted(glob.glob(f"{CHUNK_DIR}/part_*.mp4"))

for i, f in enumerate(chunks):
    print("=================================")
    print("Processing:", f)
    print("=================================")
    !python run.py -i "{f}" -m {MODE} -s {SCALE} --tiled-dit --tiled-vae {OUTPUT_DIR}

In [ ]:
import glob
files = sorted(glob.glob(f"{OUTPUT_DIR}/FlashVSR_*.mp4"))

with open("/content/list.txt","w") as f:
    for v in files:
        f.write(f"file '{v}'\n")

!ffmpeg -f concat -safe 0 -i /content/list.txt -c copy {OUTPUT_DIR}/FINAL_{INPUT}